# Segmentation

The notebook assumes one folder with images to be segmented. The path to that folder should be specified in PATH_TO_IMG_DIR. The cropped images will be saved in the "cropped_faces" folder.

In [36]:
import sys
!{sys.executable} -m pip install opencv-python supervision inference ultralytics

Defaulting to user installation because normal site-packages is not writeable


In [37]:
PATH_TO_IMG_DIR = "images"

In [39]:
# load libraries
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import supervision
from PIL import Image
import cv2
import os

# download model
model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")

# load model
model = YOLO(model_path)

for i, filename in enumerate(os.listdir(PATH_TO_IMG_DIR)):
    
    if filename.lower().endswith((".png", ".jpg", ".jpeg")) is False:
        continue

    image_path = os.path.join(PATH_TO_IMG_DIR, filename)
    image = cv2.imread(image_path)
    results = model(image)[0]
    detections = supervision.Detections.from_ultralytics(results)

    #box_annotator = supervision.BoxAnnotator()
    #label_annotator = supervision.LabelAnnotator()

    #annotated_image = box_annotator.annotate(scene=image, detections=detections)
    #annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

    h, w = image.shape[:2]

    boxes = results.boxes.xyxy.cpu().numpy()

    os.makedirs("cropped_faces", exist_ok=True)

    for j, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)

        # box width/height
        bw = x2 - x1
        bh = y2 - y1

        # padding
        pad_x = int(0.5 * bw)
        pad_y_down = int(0.3 * bh)
        pad_y_up = int(0.5 * bh)

        # expand box safely
        x1_new = max(0, x1 - pad_x)
        y1_new = max(0, y1 - pad_y_up)
        x2_new = min(w, x2 + pad_x)
        y2_new = min(h, y2 + pad_y_down)

        face_crop = image[y1_new:y2_new, x1_new:x2_new]

        cv2.imwrite(f"cropped_faces/face_{i}_{j}.jpg", face_crop)


0: 512x640 6 FACEs, 72.3ms
Speed: 1.5ms preprocess, 72.3ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 640)

0: 640x512 2 FACEs, 92.8ms
Speed: 1.8ms preprocess, 92.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 512)

0: 640x480 2 FACEs, 73.3ms
Speed: 1.9ms preprocess, 73.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 480)

0: 640x448 1 FACE, 70.9ms
Speed: 1.6ms preprocess, 70.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)

0: 480x640 1 FACE, 71.4ms
Speed: 1.3ms preprocess, 71.4ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 384x640 1 FACE, 63.0ms
Speed: 1.2ms preprocess, 63.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 640x640 1 FACE, 89.0ms
Speed: 2.4ms preprocess, 89.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
